# Best Parameter

In [13]:
import json 
import os
import numpy as np 

path = "./folder_results_mnist_results_20260518_011639/"

files = os.listdir(path=path)

print(files)

json_file = None

with open(os.path.join(path, "model_results.json"), 'r', encoding='utf-8') as f:
    json_file = json.load(f)

['neural-9-weights.npz', 'conv2d-1-bias.npz', 'conv2d-3-weights.npz', 'conv2d-3-bias.npz', 'model_results.json', 'conv2d-5-weights.npz', 'neural-8-bias.npz', 'conv2d-1-weights.npz', 'neural-8-weights.npz', 'neural-9-bias.npz', 'conv2d-5-bias.npz']


In [14]:
layer_map = ['neural', 'conv2d', 'pool', 'flatten']

model_conf = json_file['configuration']
len(model_conf)

10

In [15]:
weights_bias = {}

for i in model_conf:
    has_forward = i['forward']
    model_name = i['name']
    if has_forward:
        weights = os.path.join(path, i['wparam'])
        bias = os.path.join(path, i['bparam'])
        
        w_matrix = np.load(weights)['weights']
        b_matrix = np.load(bias)['bias']
        
        print(w_matrix.shape, b_matrix.shape, model_name)
        
        weights_bias[model_name] = {
            "w": w_matrix,
            "b": b_matrix
        }
        
    else:
        weights_bias[model_name] = None

(8, 1, 3, 3) (8,) conv2d-1
(16, 8, 3, 3) (16,) conv2d-3
(32, 16, 3, 3) (32,) conv2d-5
(32, 16) (1, 16) neural-8
(16, 10) (1, 10) neural-9


In [16]:
conv1_weights = weights_bias['conv2d-1']['w']
conv1_bias = weights_bias['conv2d-1']['b']

conv2_weights = weights_bias['conv2d-3']['w']
conv2_bias = weights_bias['conv2d-3']['b']

conv3_weights = weights_bias['conv2d-5']['w']
conv3_bias = weights_bias['conv2d-5']['b']

In [17]:
neural1_weights = weights_bias["neural-8"]["w"]
neural1_bias = weights_bias["neural-8"]["b"]

neural2_weights = weights_bias["neural-9"]["w"]
neural2_bias = weights_bias["neural-9"]["b"]

# Quantitized Model Parameter

## Conv1

In [18]:
import numpy as np
import os

OUT_DIR = "rom_mem"
os.makedirs(OUT_DIR, exist_ok=True)

FRAC_BITS = 14
W_BITS = 16
B_BITS = 16

def quantize_signed(x, bits, frac_bits):
    scale = 1 << frac_bits

    q = np.round(x * scale).astype(np.int64)

    min_q = -(1 << (bits - 1))
    max_q =  (1 << (bits - 1)) - 1

    sat_low = np.sum(q < min_q)
    sat_high = np.sum(q > max_q)

    q = np.clip(q, min_q, max_q)

    return q, sat_low, sat_high

def int_to_hex_twos_complement(v, bits):
    if v < 0:
        v = (1 << bits) + v

    hex_digits = bits // 4
    return f"{v:0{hex_digits}X}"

def save_mem(filename, q, bits):
    q_flat = q.flatten()

    with open(filename, "w") as f:
        for value in q_flat:
            f.write(int_to_hex_twos_complement(int(value), bits) + "\n")

    print(f"saved: {filename}")
    print("total:", len(q_flat))

# =========================
# LOAD CONV1 PARAMETER
# =========================

w1 = conv1_weights
b1 = conv1_bias

print("w1 shape:", w1.shape)
print("b1 shape:", b1.shape)

# Pastikan shape cocok dengan block1_weight_rom
# Harusnya: (8, 1, 3, 3)
assert w1.shape == (8, 1, 3, 3), f"w1 shape salah: {w1.shape}"
assert b1.shape == (8,), f"b1 shape salah: {b1.shape}"

# Karena input training kamu pixel / 255,
# maka hanya conv1 weight yang dibagi 255.
w1_hw = w1 / 255.0

# Bias tidak dibagi 255
b1_hw = b1

# Quantize ke signed Q?.14
q_w1, sat_w_low, sat_w_high = quantize_signed(w1_hw, W_BITS, FRAC_BITS)
q_b1, sat_b_low, sat_b_high = quantize_signed(b1_hw, B_BITS, FRAC_BITS)

# Karena shape w1 adalah:
# (out_channel, in_channel, ky, kx)
# maka flatten biasa sudah cocok:
# filter -> channel -> ky -> kx
save_mem(f"{OUT_DIR}/conv1_w.mem", q_w1, W_BITS)
save_mem(f"{OUT_DIR}/conv1_b.mem", q_b1, B_BITS)

print("Weight saturation low :", sat_w_low)
print("Weight saturation high:", sat_w_high)
print("Bias saturation low   :", sat_b_low)
print("Bias saturation high  :", sat_b_high)

print("q_w1 min/max:", q_w1.min(), q_w1.max())
print("q_b1 min/max:", q_b1.min(), q_b1.max())

w1 shape: (8, 1, 3, 3)
b1 shape: (8,)
saved: rom_mem/conv1_w.mem
total: 72
saved: rom_mem/conv1_b.mem
total: 8
Weight saturation low : 0
Weight saturation high: 0
Bias saturation low   : 0
Bias saturation high  : 0
q_w1 min/max: -65 68
q_b1 min/max: -4307 9


## Conv2

In [19]:
import numpy as np
import os

OUT_DIR = "rom_mem"
os.makedirs(OUT_DIR, exist_ok=True)

FRAC_BITS = 14
W_BITS = 16
B_BITS = 16

def quantize_signed(x, bits, frac_bits):
    scale = 1 << frac_bits

    q = np.round(x * scale).astype(np.int64)

    min_q = -(1 << (bits - 1))
    max_q =  (1 << (bits - 1)) - 1

    sat_low = np.sum(q < min_q)
    sat_high = np.sum(q > max_q)

    q = np.clip(q, min_q, max_q)

    return q, sat_low, sat_high

def int_to_hex_twos_complement(v, bits):
    if v < 0:
        v = (1 << bits) + v

    hex_digits = bits // 4
    return f"{v:0{hex_digits}X}"

def save_mem(filename, q, bits):
    q_flat = q.flatten()

    with open(filename, "w") as f:
        for value in q_flat:
            f.write(int_to_hex_twos_complement(int(value), bits) + "\n")

    print(f"saved: {filename}")
    print("total:", len(q_flat))

# =========================
# LOAD CONV2 PARAMETER
# =========================

w2 = conv2_weights

b2 = conv2_bias

print("w2 shape:", w2.shape)
print("b2 shape:", b2.shape)

# Harus cocok dengan conv2_partial:
# 16 output channel, 8 input channel, 3x3 kernel
assert w2.shape == (16, 8, 3, 3), f"w2 shape salah: {w2.shape}"
assert b2.shape == (16,), f"b2 shape salah: {b2.shape}"

# Conv2 TIDAK dibagi 255.
# /255 hanya untuk conv1.
w2_hw = w2
b2_hw = b2

q_w2, sat_w_low, sat_w_high = quantize_signed(w2_hw, W_BITS, FRAC_BITS)
q_b2, sat_b_low, sat_b_high = quantize_signed(b2_hw, B_BITS, FRAC_BITS)

# w2 shape:
# (out_channel, in_channel, ky, kx)
#
# flatten menghasilkan urutan:
# oc0 ic0 ky0 kx0
# oc0 ic0 ky0 kx1
# ...
# oc0 ic1 ky0 kx0
# ...
# oc1 ic0 ky0 kx0
# ...
save_mem(f"{OUT_DIR}/conv2_w.mem", q_w2, W_BITS)
save_mem(f"{OUT_DIR}/conv2_b.mem", q_b2, B_BITS)

print("Weight saturation low :", sat_w_low)
print("Weight saturation high:", sat_w_high)
print("Bias saturation low   :", sat_b_low)
print("Bias saturation high  :", sat_b_high)

print("q_w2 min/max:", q_w2.min(), q_w2.max())
print("q_b2 min/max:", q_b2.min(), q_b2.max())

print("Expected conv2_w.mem lines:", 16 * 8 * 3 * 3)
print("Expected conv2_b.mem lines:", 16)

w2 shape: (16, 8, 3, 3)
b2 shape: (16,)
saved: rom_mem/conv2_w.mem
total: 1152
saved: rom_mem/conv2_b.mem
total: 16
Weight saturation low : 0
Weight saturation high: 0
Bias saturation low   : 0
Bias saturation high  : 0
q_w2 min/max: -8030 10823
q_b2 min/max: -6226 2164
Expected conv2_w.mem lines: 1152
Expected conv2_b.mem lines: 16


## Conv3

In [20]:
import numpy as np
import os

# ============================================================
# CONFIG
# ============================================================

RESULT_DIR = "results/folder_results_mnist_results_20260518_011639"
OUT_DIR = "rom_mem"
os.makedirs(OUT_DIR, exist_ok=True)

FRAC_BITS = 14
W_BITS = 16
B_BITS = 16

CONV3_W_PATH = f"{RESULT_DIR}/conv2d-5-weights.npz"
CONV3_B_PATH = f"{RESULT_DIR}/conv2d-5-bias.npz"

OUT_W_MEM = f"{OUT_DIR}/conv3_w.mem"
OUT_B_MEM = f"{OUT_DIR}/conv3_b.mem"


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def quantize_signed(x, bits, frac_bits):
    scale = 1 << frac_bits

    q = np.round(x * scale).astype(np.int64)

    min_q = -(1 << (bits - 1))
    max_q =  (1 << (bits - 1)) - 1

    sat_low = np.sum(q < min_q)
    sat_high = np.sum(q > max_q)

    q = np.clip(q, min_q, max_q)

    return q.astype(np.int64), sat_low, sat_high


def int_to_hex_twos_complement(v, bits):
    if v < 0:
        v = (1 << bits) + v

    hex_digits = bits // 4
    return f"{v:0{hex_digits}X}"


def save_mem_signed(filename, q_array, bits):
    q_flat = q_array.flatten()

    with open(filename, "w") as f:
        for value in q_flat:
            f.write(int_to_hex_twos_complement(int(value), bits) + "\n")

    print(f"saved: {filename}")
    print("total lines:", len(q_flat))


def conv3_addr(oc, ic, ky, kx):
    """
    Cocok dengan:
    addr = oc * (16*9) + ic * 9 + ky * 3 + kx
    """
    return oc * (16 * 9) + ic * 9 + ky * 3 + kx


# ============================================================
# LOAD CONV3 PARAMETER
# ============================================================

w3 = conv3_weights

b3 = conv3_bias

print("w3 shape:", w3.shape)
print("b3 shape:", b3.shape)

assert w3.shape == (32, 16, 3, 3), f"w3 shape salah: {w3.shape}"
assert b3.shape == (32,), f"b3 shape salah: {b3.shape}"


# ============================================================
# QUANTIZATION
# ============================================================

# Conv3 TIDAK dibagi 255.
# /255 hanya untuk conv1 weight.
w3_hw = w3
b3_hw = b3

q_w3, sat_w_low, sat_w_high = quantize_signed(
    w3_hw,
    bits=W_BITS,
    frac_bits=FRAC_BITS
)

q_b3, sat_b_low, sat_b_high = quantize_signed(
    b3_hw,
    bits=B_BITS,
    frac_bits=FRAC_BITS
)


# ============================================================
# SAVE .MEM
# ============================================================

# Karena shape w3 = (out_channel, input_channel, ky, kx),
# flatten biasa sudah menghasilkan urutan:
# oc -> ic -> ky -> kx
save_mem_signed(OUT_W_MEM, q_w3, W_BITS)
save_mem_signed(OUT_B_MEM, q_b3, B_BITS)


# ============================================================
# REPORT
# ============================================================

print()
print("===================================================")
print("CONV3 PARAMETER DUMP REPORT")
print("===================================================")
print("FRAC_BITS:", FRAC_BITS)
print("W_BITS   :", W_BITS)
print("B_BITS   :", B_BITS)
print()

print("Expected conv3_w.mem lines:", 32 * 16 * 3 * 3)
print("Expected conv3_b.mem lines:", 32)
print()

print("Weight float min/max:", w3_hw.min(), w3_hw.max())
print("Weight quant min/max:", q_w3.min(), q_w3.max())
print("Weight saturation low :", sat_w_low)
print("Weight saturation high:", sat_w_high)
print()

print("Bias float min/max:", b3_hw.min(), b3_hw.max())
print("Bias quant min/max:", q_b3.min(), q_b3.max())
print("Bias saturation low :", sat_b_low)
print("Bias saturation high:", sat_b_high)
print()


# ============================================================
# SANITY CHECK ADDRESS MAPPING
# ============================================================

print("===================================================")
print("ADDRESS MAPPING CHECK")
print("===================================================")

checks = [
    (0, 0, 0, 0),
    (0, 0, 0, 1),
    (0, 0, 0, 2),
    (0, 0, 1, 0),
    (0, 7, 2, 2),
    (0, 8, 0, 0),
    (0, 15, 2, 2),
    (1, 0, 0, 0),
    (31, 15, 2, 2),
]

q_flat = q_w3.flatten()

for oc, ic, ky, kx in checks:
    addr = conv3_addr(oc, ic, ky, kx)
    from_array = q_w3[oc, ic, ky, kx]
    from_flat = q_flat[addr]

    print(
        f"oc={oc:2d}, ic={ic:2d}, ky={ky}, kx={kx} "
        f"-> addr={addr:4d}, q={from_array:7d}, flat={from_flat:7d}"
    )

    assert from_array == from_flat

print()
print("conv3_w.mem dan conv3_b.mem siap dipakai.")

w3 shape: (32, 16, 3, 3)
b3 shape: (32,)
saved: rom_mem/conv3_w.mem
total lines: 4608
saved: rom_mem/conv3_b.mem
total lines: 32

CONV3 PARAMETER DUMP REPORT
FRAC_BITS: 14
W_BITS   : 16
B_BITS   : 16

Expected conv3_w.mem lines: 4608
Expected conv3_b.mem lines: 32

Weight float min/max: -0.5716528778355754 0.4804287040209989
Weight quant min/max: -9366 7871
Weight saturation low : 0
Weight saturation high: 0

Bias float min/max: -0.13559953435270944 0.15367000378293352
Bias quant min/max: -2222 2518
Bias saturation low : 0
Bias saturation high: 0

ADDRESS MAPPING CHECK
oc= 0, ic= 0, ky=0, kx=0 -> addr=   0, q=  -1530, flat=  -1530
oc= 0, ic= 0, ky=0, kx=1 -> addr=   1, q=  -1032, flat=  -1032
oc= 0, ic= 0, ky=0, kx=2 -> addr=   2, q=   1418, flat=   1418
oc= 0, ic= 0, ky=1, kx=0 -> addr=   3, q=    569, flat=    569
oc= 0, ic= 7, ky=2, kx=2 -> addr=  71, q=  -1169, flat=  -1169
oc= 0, ic= 8, ky=0, kx=0 -> addr=  72, q=    555, flat=    555
oc= 0, ic=15, ky=2, kx=2 -> addr= 143, q=   19

## Dense FC1 to FC2

In [21]:
import numpy as np
import os

OUT_DIR = "rom_mem"
os.makedirs(OUT_DIR, exist_ok=True)

FRAC_BITS = 14
W_BITS = 16
B_BITS = 32   # IMPORTANT: fc1_fc2_2dsp.v expects 32-bit bias memories

def quantize_signed(x, bits, frac_bits):
    q = np.round(x * (1 << frac_bits)).astype(np.int64)
    min_q = -(1 << (bits - 1))
    max_q =  (1 << (bits - 1)) - 1
    sat_low = int(np.sum(q < min_q))
    sat_high = int(np.sum(q > max_q))
    q = np.clip(q, min_q, max_q)
    return q.astype(np.int64), sat_low, sat_high

def to_twos_hex(v, bits):
    v = int(v)
    if v < 0:
        v = (1 << bits) + v
    return f"{v:0{bits//4}X}"

def save_mem(path, q, bits):
    flat = q.reshape(-1)
    with open(path, "w") as f:
        for v in flat:
            f.write(to_twos_hex(v, bits) + "\n")
    print(f"saved: {path} lines={len(flat)} bits={bits}")

def report(name, arr, q, lo, hi):
    print(f"\n{name}")
    print("  shape:", arr.shape)
    print("  float min/max:", float(np.min(arr)), float(np.max(arr)))
    print("  quant min/max:", int(np.min(q)), int(np.max(q)))
    print("  saturation low/high:", lo, hi)

w4 = neural1_weights  # shape (32, 16) = input, hidden
b4 = neural1_bias     # shape (1, 16)
w5 = neural2_weights  # shape (16, 10) = hidden, class
b5 = neural2_bias     # shape (1, 10)

print("w4,b4,w5,b5 shapes:", w4.shape, b4.shape, w5.shape, b5.shape)
assert w4.shape == (32, 16)
assert b4.shape == (1, 16)
assert w5.shape == (16, 10)
assert b5.shape == (1, 10)

# Verilog indexing:
#   fc1_w[hidden_neuron * FC1_IN + input_idx]
#   fc2_w[out_class     * FC1_OUT + hidden_idx]
# Therefore transpose Python matrices from input-major to output-major.
w4_rom = w4.T                 # (16, 32)
w5_rom = w5.T                 # (10, 16)
b4_rom = b4.reshape(-1)       # (16,)
b5_rom = b5.reshape(-1)       # (10,)

q_w4, sw4_l, sw4_h = quantize_signed(w4_rom, W_BITS, FRAC_BITS)
q_b4, sb4_l, sb4_h = quantize_signed(b4_rom, B_BITS, FRAC_BITS)
q_w5, sw5_l, sw5_h = quantize_signed(w5_rom, W_BITS, FRAC_BITS)
q_b5, sb5_l, sb5_h = quantize_signed(b5_rom, B_BITS, FRAC_BITS)

save_mem(f"{OUT_DIR}/fc1_w.mem", q_w4, W_BITS)
save_mem(f"{OUT_DIR}/fc1_b.mem", q_b4, B_BITS)
save_mem(f"{OUT_DIR}/fc2_w.mem", q_w5, W_BITS)
save_mem(f"{OUT_DIR}/fc2_b.mem", q_b5, B_BITS)

print("\nExpected lines:")
print("  fc1_w.mem", 16*32)
print("  fc1_b.mem", 16)
print("  fc2_w.mem", 10*16)
print("  fc2_b.mem", 10)

report("FC1 weight", w4_rom, q_w4, sw4_l, sw4_h)
report("FC1 bias", b4_rom, q_b4, sb4_l, sb4_h)
report("FC2 weight", w5_rom, q_w5, sw5_l, sw5_h)
report("FC2 bias", b5_rom, q_b5, sb5_l, sb5_h)

# Print first few bias lines so you can verify they are 8 hex digits.
print("\nBias memory preview must be 8 hex digits each:")
print("  fc1_b first 5:", [to_twos_hex(v, B_BITS) for v in q_b4[:5]])
print("  fc2_b first 5:", [to_twos_hex(v, B_BITS) for v in q_b5[:5]])

w4,b4,w5,b5 shapes: (32, 16) (1, 16) (16, 10) (1, 10)
saved: rom_mem/fc1_w.mem lines=512 bits=16
saved: rom_mem/fc1_b.mem lines=16 bits=32
saved: rom_mem/fc2_w.mem lines=160 bits=16
saved: rom_mem/fc2_b.mem lines=10 bits=32

Expected lines:
  fc1_w.mem 512
  fc1_b.mem 16
  fc2_w.mem 160
  fc2_b.mem 10

FC1 weight
  shape: (16, 32)
  float min/max: -0.8352600612450687 0.8775102506031276
  quant min/max: -13685 14377
  saturation low/high: 0 0

FC1 bias
  shape: (16,)
  float min/max: -0.12749861729619108 0.16404383492023547
  quant min/max: -2089 2688
  saturation low/high: 0 0

FC2 weight
  shape: (10, 16)
  float min/max: -1.3418329406431884 1.03014726468744
  quant min/max: -21985 16878
  saturation low/high: 0 0

FC2 bias
  shape: (10,)
  float min/max: -0.28217069798953803 0.23237672595760003
  quant min/max: -4623 3807
  saturation low/high: 0 0

Bias memory preview must be 8 hex digits each:
  fc1_b first 5: ['0000019F', '000005A1', '00000104', '00000078', '00000A80']
  fc2_b fir